In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Physical Parameters ---
c = 3e8                  # Speed of light (m/s)
nu = 100e6               # Frequency: 1.4 GHz (L-band)
b = 14.0                # Baseline length: 100 meters (East-West)
alpha = 12.0             # Source Right Ascension (hours)
S_0 = 1.0               # Peak source flux (Janskys)

# Angular sizes (in radians, exaggerated for visual clarity)
sigma_beam = np.deg2rad(5.0)  # 5-degree beam width
sigma_src = np.deg2rad(0.5)   # 0.5-degree source size

# --- Derived Parameters ---
u = (b * nu) / c
Sigma2 = sigma_beam**2 + 2 * sigma_src**2
sigma_eff2 = (2 * sigma_beam**2 * sigma_src**2) / Sigma2
S_total = S_0 * (2 * np.pi * sigma_eff2)

# --- Time Axis (LST in hours) ---
# Simulating 4 hours of drift across the meridian
lst_hours = np.linspace(alpha - 6, alpha + 6, 1000)
H_rad = (lst_hours - alpha) * (np.pi / 12) # Convert LST-alpha to radians

# --- The Analytical Equation ---
l0 = np.sin(H_rad)

# 1. Attenuation Envelope
A = np.exp(- (l0**2) / Sigma2)
# print("A shape:", A.shape, A)

# 2. Resolution Term (Constant scalar)
R = np.exp(-2 * np.pi**2 * sigma_eff2 * u**2)
# print("R shape:", R.shape, R)

# 3. Phase Term
phase_argument = -2 * np.pi * u * l0 * (sigma_beam**2 / Sigma2)
Phi = np.exp(1j * phase_argument)

# Final Visibility
V = S_total * A * R * Phi

# --- Plotting ---
plt.figure(figsize=(12, 6))

# Plot the absolute amplitude (The Envelope)
# plt.plot(lst_hours, np.abs(V), 'r--', lw=2, label='Amplitude Envelope (Beam Pattern)')

# Plot the real part of the visibility (The Fringes)
# plt.plot(lst_hours, 1e-5*np.angle(V), 'b-', lw=1.5, alpha=0.5, label='Phase of Visibility - The Fringes')

# Optional: Plot the phase term separately to visualize its behavior
plt.plot(lst_hours, np.abs(Phi), 'g-', lw=1.5, alpha=0.8, label='Abs(Phase) - The Phase Term')

# Optional: Plot the phase term separately to visualize its behavior
# plt.plot(lst_hours, 1e-5*np.angle(Phi), 'orange', lw=1.5, alpha=0.5, linestyle='--', label='Angle(Phase) - The Phase Term')

# plt.plot(lst_hours, 1e-5*np.angle(R), 'red', lw=1.5, alpha=0.5, linestyle='--', label='Angle(Phase) - The Phase Term')
# plt.plot(lst_hours, 1e-5*np.angle(A), 'red', lw=1.5, alpha=0.5, linestyle='--', label='Angle(Phase) - The Phase Term')

plt.title(f"Interferometer Drift Scan (Baseline = {b}m, Freq = {nu/1e9} GHz)")
plt.xlabel("Local Sidereal Time (Hours)")
plt.ylabel("Visibility (Janskys)")
plt.axvline(alpha, color='k', linestyle=':', label='Transit (Zenith)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Instrument Parameters ---
c = 3e8
nu = 200e6               # 1.4 GHz
b = 100.0                # 100m East-West Baseline
u = (b * nu) / c
sigma_beam = np.deg2rad(5.0)

# --- Define Two Sources ---
# Source 1: Strong, exactly at zenith
alpha1 = 12.0  
delta1 = 0.0
S1 = 4.0      
sigma_src1 = np.deg2rad(0.1)

# Source 2: Weaker, slightly ahead in RA, slightly North in Dec
alpha2 = 6 # Inside the 5-deg beam at the same time!
delta2 = np.deg2rad(1.5) # m != 0
S2 = 2.0       
sigma_src2 = np.deg2rad(0.1)

# --- Time Axis ---
lst_hours = np.linspace(1, 16, 2000)

def get_visibility(alpha, delta_rad, S0, sig_src):
    # Calculate geometric terms
    H_rad = (lst_hours - alpha) * (np.pi / 12)
    l = np.sin(H_rad)
    m = np.sin(delta_rad)
    
    # Calculate effective widths
    Sigma2 = sigma_beam**2 + 2 * sig_src**2
    sig_eff2 = (2 * sigma_beam**2 * sig_src**2) / Sigma2
    Stot = S0 * (2 * np.pi * sig_eff2)
    
    # RIME Components
    A = np.exp(- (l**2 + m**2) / Sigma2)
    R = np.exp(-2 * np.pi**2 * sig_eff2 * u**2)
    Phi = np.exp(-2j * np.pi * u * l * (sigma_beam**2 / Sigma2))
    
    return Stot * A * R * Phi, Phi

# --- Superposition (The Linear Sum) ---
V1, Phi1 = get_visibility(alpha1, delta1, S1, sigma_src1)
V2, Phi2 = get_visibility(alpha2, delta2, S2, sigma_src2)
V_total = V1 + V2

# --- Plotting ---
plt.figure(figsize=(12, 6))
plt.plot(lst_hours, np.abs(V_total), 'r--', lw=2.5, label='Total Amplitude Envelope (The Beat Pattern)')
# plt.plot(lst_hours, np.real(V_total), 'b-', lw=1, alpha=0.7, label='Real(V_total) - Superimposed Fringes')
# plt.plot(lst_hours, np.angle(V_total), 'b-', lw=1, alpha=0.7, label='Phase of Total Visibility')

# plt.plot(lst_hours, 1e-5*np.real(Phi1), 'b-', lw=1, alpha=0.7, label='Real(Phi1) - Source 1 Phase')
# plt.plot(lst_hours, 1e-5*np.real(Phi2), 'g-', lw=1, alpha=0.7, label='Real(Phi2) - Source 2 Phase')

plt.title("Interferometer Drift Scan: Two Sources Beating")
plt.xlabel("Local Sidereal Time (Hours)")
plt.ylabel("Correlated Flux (Visibility)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- 1. Instrument Parameters ---
c = 3e8
nu = 100e6               # 1.4 GHz
b = 100.0                # 100m East-West Baseline
u = (b * nu) / c         # Baseline in wavelengths
sigma_beam = np.deg2rad(5.0)

# --- 2. Define Two Sources ---
alpha1, delta1, S1, sigma_src1 = 12.0,  0.0,             10.0, np.deg2rad(0.1)
alpha2, delta2, S2, sigma_src2 = 6.15, np.deg2rad(1.5), 7.0,  np.deg2rad(0.1)

# --- 3. Time Axis ---
lst_hours = np.linspace(1.0, 13.0, 1000)

# --- 4. Visibility Functions ---
def get_visibility(alpha, delta_rad, S0, sig_src, baseline_u):
    H_rad = (lst_hours - alpha) * (np.pi / 12)
    l = np.sin(H_rad)
    m = np.sin(delta_rad)
    
    Sigma2 = sigma_beam**2 + 2 * sig_src**2
    sig_eff2 = (2 * sigma_beam**2 * sig_src**2) / Sigma2
    Stot = S0 * (2 * np.pi * sig_eff2)
    
    A = np.exp(- (l**2 + m**2) / Sigma2)
    R = np.exp(-2 * np.pi**2 * sig_eff2 * baseline_u**2)
    Phi = np.exp(-2j * np.pi * baseline_u * l * (sigma_beam**2 / Sigma2))
    
    return Stot * A * R * Phi

# Calculate Cross-Correlation (u = 466.6 wavelengths)
V1_cross = get_visibility(alpha1, delta1, S1, sigma_src1, u)
V2_cross = get_visibility(alpha2, delta2, S2, sigma_src2, u)
V_total_cross = V1_cross + V2_cross

# Calculate Autocorrelation (u = 0)
V1_auto = get_visibility(alpha1, delta1, S1, sigma_src1, 0.0)
V2_auto = get_visibility(alpha2, delta2, S2, sigma_src2, 0.0)
V_total_auto = np.real(V1_auto + V2_auto) # Real because Phase is 0


# --- 5. Calculate Power ---
# Cross-Correlation Power (Squared Magnitude)
Power_cross = np.abs(V_total_cross)**2

# Autocorrelation Power is just the zero-baseline measurement
Power_auto = V_total_auto 


# --- 6. Plotting ---
plt.figure(figsize=(10, 6))

# Plot the Autocorrelation (Total Power)
# plt.plot(lst_hours, Power_auto, 'k--', lw=2, 
#          label='Autocorrelation (Zero-Baseline Power)\n$|V_1|_{u=0} + |V_2|_{u=0}$')

# Plot the Cross-Correlation Power
plt.plot(lst_hours, Power_cross, 'r-', lw=2, 
         label='Cross-Correlation Power (Fringe Power)\n$|V_1 + V_2|^2$')

plt.title(f"Visibility Power vs LST at {nu/1e9} GHz")
plt.xlabel("Local Sidereal Time (Hours)")
plt.ylabel("Power (Arbitrary Units)")

# Annotate the source transit times
plt.axvline(alpha1, color='gray', linestyle=':', label='Transit Source 1')
plt.axvline(alpha2, color='gray', linestyle='-.', label='Transit Source 2')

plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- 1. Define the numerical grids ---
x0_vals = np.linspace(-15, 15, 500)  # The shift parameter (e.g., source moving)
x = np.linspace(-50, 50, 2000)       # The integration variable (e.g., the sky)
dx = x[1] - x[0]

# --- 2. Define the functions ---
def f(x):
    return np.exp(-x**2 / 2.0)       # A simple Gaussian

y_vals = np.array([1.0, 1.2, 1.5])   # Multiple 'y' frequencies
g_vals = np.array([1.0, 0.8, 0.5])   # Weights g(y)

# Arrays to store the results
P1 = np.zeros_like(x0_vals)
P2 = np.zeros_like(x0_vals)

# --- 3. Numerical Integration ---
for i, x0 in enumerate(x0_vals):
    # The shifted function
    fx = f(x - x0)
    
    # Integral 1: ( integral f(x-x0) e^(ix) dx )^2
    int1 = np.sum(fx * np.exp(1j * x)) * dx
    P1[i] = np.abs(int1)**2
    
    # Integral 2: ( sum_y integral f(x-x0) g(y) e^(ixy) dx )^2
    int2 = 0
    for y, gy in zip(y_vals, g_vals):
        int_y = np.sum(fx * np.exp(1j * x * y)) * dx
        int2 += gy * int_y
    P2[i] = np.abs(int2)**2

# --- 4. Plotting ---
plt.figure(figsize=(10, 6))

plt.plot(x0_vals, P1, 'k-', lw=2.5, label='Eq 1: Single Mode Power $|\\int f(x-x_0) e^{ix} dx|^2$')
plt.plot(x0_vals, P2, 'r-', lw=2, label='Eq 2: Multi-Mode Power $|\\sum_y g(y) \\dots|^2$')

plt.title("The Consequence of Taking the Power (Squaring the Magnitude)")
plt.xlabel("Shift Parameter $x_0$ (e.g., Source Position)")
plt.ylabel("Squared Magnitude (Power)")
plt.legend()
plt.grid(True)
plt.ylim(0, max(P2)*1.1)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- 1. Instrument Parameters ---
c = 3e8
nu_base = 100e6          # Base Frequency: 100 MHz
b = 10.0                # 100m East-West Baseline
u_base = (b * nu_base)/c # Baseline in wavelengths at 100 MHz
sigma_beam = np.deg2rad(5.0)

# --- 2. Define Two Sources ---
# Notice: You placed these at 12.0 and 6.15 hours. 
# They are far apart, so they will pass through the zenith one at a time.
alpha1, delta1, S1, sigma_src1 = 12.0,  0.0,             4.0, np.deg2rad(5)
alpha2, delta2, S2, sigma_src2 = 8.15, np.deg2rad(1.5), 2.0,  np.deg2rad(5)

# --- 3. Time Axis ---
# Increased to 3000 points to smoothly resolve the sharp multi-frequency peaks
lst_hours = np.linspace(1.0, 13.0, 3000)

# --- 4. Visibility Functions ---
def get_visibility(alpha, delta_rad, S0, sig_src, baseline_u):
    H_rad = (lst_hours - alpha) * (np.pi / 12)
    l = np.sin(H_rad)
    m = np.sin(delta_rad)
    
    Sigma2 = sigma_beam**2 + 2 * sig_src**2
    sig_eff2 = (2 * sigma_beam**2 * sig_src**2) / Sigma2
    Stot = S0 * (2 * np.pi * sig_eff2)
    
    A = np.exp(- (l**2 + m**2) / Sigma2)
    R = np.exp(-2 * np.pi**2 * sig_eff2 * baseline_u**2)
    Phi = np.exp(-2j * np.pi * baseline_u * l * (sigma_beam**2 / Sigma2))
    
    return Stot * A * R * Phi

# Calculate Single-Frequency Cross-Correlation (at 100 MHz)
V1_cross = get_visibility(alpha1, delta1, S1, sigma_src1, u_base)
V2_cross = get_visibility(alpha2, delta2, S2, sigma_src2, u_base)
V_total_single_freq = V1_cross + V2_cross


# =====================================================================
# --- NEW SECTION: Summing over all frequencies THEN taking Power ---
# =====================================================================

# Define a wide frequency band (e.g., 51 channels from 100 MHz to 150 MHz)
freqs = np.linspace(100e6, 150e6, 51)
V_total_multi_freq = np.zeros_like(lst_hours, dtype=complex)

for f in freqs:
    u_f = (b * f) / c
    # Calculate complex visibilities for this specific frequency
    V1_f = get_visibility(alpha1, delta1, S1, sigma_src1, u_f)
    V2_f = get_visibility(alpha2, delta2, S2, sigma_src2, u_f)
    
    # Coherently sum (superimpose) the frequencies
    V_total_multi_freq += (V1_f + V2_f)

# Normalize by number of channels so the maximum amplitudes match the single-freq case
V_total_multi_freq /= len(freqs)


# --- 5. Calculate Power ---
# 1. Power of the single frequency
Power_single_freq = np.abs(V_total_single_freq)**2

# 2. Power AFTER summing across the frequency band
Power_multi_freq = np.abs(V_total_multi_freq)**2


# --- 6. Plotting ---
plt.figure(figsize=(12, 6))

# Plot the Single Frequency Power
plt.plot(lst_hours, Power_single_freq, 'k--', lw=2, alpha=0.6,
         label='Single Freq Power (100 MHz)\n$|V(100MHz)|^2$')

# Plot the Multi-Frequency Power
plt.plot(lst_hours, Power_multi_freq, 'r-', lw=2, 
         label='Multi-Freq Power (100-150 MHz)\n$|\\sum V(\\nu)|^2$')

plt.title("The Effect of Summing Frequencies (Bandwidth Smearing / Delay Tracking)")
plt.xlabel("Local Sidereal Time (Hours)")
plt.ylabel("Squared Magnitude (Power)")

# Annotate the source transit times
plt.axvline(alpha1, color='gray', linestyle=':', label='Transit Source 1')
plt.axvline(alpha2, color='gray', linestyle='-.', label='Transit Source 2')

plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- 1. Physical & Instrument Parameters ---
c = 3e8                    # Speed of light (m/s)
nu_0 = 150e6               # Center Frequency: 150 MHz (Typical for delay tracking)
B = 100e6                   # Bandwidth: 30 MHz
b = 100.0                  # East-West Baseline: 100 meters

# Angular definitions (in radians)
sigma_beam = np.deg2rad(10.0)  # 10-degree primary beam (wide at low frequencies)
sigma_src = np.deg2rad(0.1)    # 0.1-degree source (mostly point-like)
S_0 = 2.0                     # Source Peak Flux (Janskys)

# Source Position
alpha = 12.0                   # Right Ascension (Transit time in LST hours)

# --- 2. Time Axis (24 Hours of LST) ---
# Using 5000 points to ensure we smoothly resolve the sinc wiggles
lst_hours = np.linspace(0.0, 24.0, 5000)

# Calculate direction cosine l_0 for an East-West drift
H_rad = (lst_hours - alpha) * (np.pi / 12.0)
l_0 = np.sin(H_rad)

# --- 3. Effective Math Parameters ---
D = sigma_beam**2 + 2 * sigma_src**2
sigma_eff2 = (sigma_beam**2 * sigma_src**2) / D
l_eff = l_0 * (sigma_beam**2 / D)

# --- 4. Calculate the 0-Delay Power Components ---
# (We square the exact analytical magnitude derived previously)

# 1. Constant Flux Scalar: S_0^2 * 2*pi*sigma_eff^2
flux_scalar = (S_0**2) * (2 * np.pi * sigma_eff2)

# 2. Primary Beam Attenuation (Squared)
beam_power = np.exp(-2 * (l_0**2) / D)

# 3. Resolution Loss (Squared constant)
res_loss_power = np.exp(-4 * np.pi**2 * sigma_eff2 * (b * nu_0 / c)**2)

# 4. The Delay Filter: B^2 * sinc^2(tau_g * B)
# Note: tau_g = (b * l_eff) / c. 
# numpy's sinc(x) is defined as sin(pi*x)/(pi*x), which perfectly matches our derivation.
tau_g = (b * l_eff) / c
sinc_power = (B**2) * (np.sinc(tau_g * B)**2)

# --- 5. Final Equations ---
# Total 0-Delay Power (with Bandwidth Smearing)
Power_0_delay = flux_scalar * beam_power * res_loss_power * sinc_power

# For comparison: Single Frequency Power Envelope (No bandwidth smearing / sinc term)
# We multiply by B^2 just to scale it to the exact same peak height for visual comparison
Power_single_freq = flux_scalar * beam_power * res_loss_power * (B**2)

# --- 6. Plotting ---
plt.figure(figsize=(12, 6))

# Plot the broad Single Frequency beam envelope
plt.plot(lst_hours, Power_single_freq, 'k--', lw=2, alpha=0.5, 
         label="Single Frequency Envelope\n(No Bandwidth / Delay Filter)")

# Plot the 0-Delay Power with the Sinc function
plt.plot(lst_hours, Power_0_delay, 'r-', lw=2.5, 
         label="0-Delay Power ($|\\tilde{V}(\\tau=0)|^2$)\n(Finite Bandwidth Sinc Filter)")

# Formatting
plt.title(f"0-Delay Power vs LST (Baseline: {b}m, Center Freq: {nu_0/1e6} MHz, BW: {B/1e6} MHz)", fontsize=14)
plt.xlabel("Local Sidereal Time (Hours)", fontsize=12)
plt.ylabel("Squared Magnitude Power (Arbitrary Units)", fontsize=12)

# Zoom in around the transit time to see the primary beam and sinc lobes clearly
plt.xlim(alpha - 3, alpha + 3)
plt.axvline(alpha, color='gray', linestyle=':', label='Transit (Zenith)')

plt.legend(fontsize=11)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- 1. Physical & Instrument Parameters ---
c = 3e8                    # Speed of light (m/s)
nu_0 = 150e6               # Center Frequency: 150 MHz 
B = 100e6                   # Bandwidth: 30 MHz
b = 10.0                  # East-West Baseline: 100 meters

# Angular definitions (in radians)
sigma_beam = np.deg2rad(10.0)  # 10-degree primary beam
sigma_src = np.deg2rad(0.1)    # 0.1-degree source size
S_0 = 1.0                     # Source Peak Flux (Janskys)

# --- 2. Define Multiple Sources ---
# 5 sources, spaced by 1.5 hours of LST
# alphas =[8.0, 8.5, 9, 9.5, 10, 10.5, 11.0, 11.5, 12, 12.5 ]
alphas =[8, 8.2, 10, 11,5, 12, 7]

# --- 3. Time Axis ---
# Simulating from LST 5.0 to 17.0 hours to frame the sources
lst_hours = np.linspace(5.0, 17.0, 5000)

# --- 4. Effective Math Parameters ---
D = sigma_beam**2 + 2 * sigma_src**2
sigma_eff2 = (sigma_beam**2 * sigma_src**2) / D

# Constants that don't depend on time
flux_amp = S_0 * np.sqrt(2 * np.pi * sigma_eff2)
res_loss = np.exp(-2 * np.pi**2 * sigma_eff2 * (b * nu_0 / c)**2)

# --- 5. Compute Visibilities ---
# Initialize arrays to hold the total complex signal and broad envelopes
V_total_0_delay = np.zeros_like(lst_hours, dtype=complex)
Power_single_freq_total = np.zeros_like(lst_hours)

for alpha in alphas:
    # 1. Geometric parameters for this specific source
    H_rad = (lst_hours - alpha) * (np.pi / 12.0)
    l_0 = np.sin(H_rad)
    l_eff = l_0 * (sigma_beam**2 / D)
    tau_g = (b * l_eff) / c
    
    # 2. Compute the Amplitude Components
    beam_amp = np.exp(- (l_0**2) / D)
    sinc_amp = B * np.sinc(tau_g * B)  # np.sinc is sin(pi*x)/(pi*x)
    
    # 3. Compute the Complex Phase (Carrier Phase)
    phase_term = np.exp(-2j * np.pi * tau_g * nu_0)
    
    # 4. Construct the Complex 0-Delay Visibility for this source
    V_source = flux_amp * beam_amp * res_loss * sinc_amp * phase_term
    
    # Coherently sum the 0-Delay visibilities
    V_total_0_delay += V_source
    
    # 5. Compute the broad Single-Frequency power envelope for visual comparison
    # (Just the flux, beam, and resolution loss squared. No sinc filter!)
    Power_single_freq_total += (flux_amp * beam_amp * res_loss * B)**2

# Calculate the final Total 0-Delay Power (Squared Magnitude of the coherent sum)
Power_0_delay_total = np.abs(V_total_0_delay)**2


# --- 6. Plotting ---
plt.figure(figsize=(14, 7))

# Plot the blended single-frequency power
plt.plot(lst_hours, Power_single_freq_total, 'k--', lw=2, alpha=0.4, 
         label="Single Frequency Envelope (What a single channel sees)\nSources are totally blended!")

# Plot the 0-Delay Power
plt.plot(lst_hours, Power_0_delay_total, 'r-', lw=2.5, 
         label="0-Delay Power ($|\\sum \\tilde{V}_k(\\tau=0)|^2$)\n(Finite Bandwidth Sinc Filter)")

# Formatting
plt.title(f"0-Delay Power for 5 Drifting Sources (Baseline: {b}m, BW: {B/1e6} MHz)", fontsize=15)
plt.xlabel("Local Sidereal Time (Hours)", fontsize=13)
plt.ylabel("Squared Magnitude Power (Arbitrary Units)", fontsize=13)

# Add vertical lines for the exact transit times
for idx, alpha in enumerate(alphas):
    if idx == 0:
        plt.axvline(alpha, color='gray', linestyle=':', label='Source Transit Times')
    else:
        plt.axvline(alpha, color='gray', linestyle=':')

plt.legend(fontsize=12, loc='upper right')
plt.grid(True, alpha=0.5)
plt.tight_layout()
plt.show()